# Phase 5 — The Efficiency Contribution
## VGGT at 224 px = 518 px Accuracy at 3× Lower Cost

Resolution sensitivity experiments (notebooks 04 and 04b) established that
VGGT's **pose accuracy is invariant to input resolution** across 224–518 px.
This notebook quantifies and presents that finding as the primary thesis
contribution.

**Core claim**  
> Running VGGT at 224 px instead of its native 518 px incurs **zero ATE
> penalty** while reducing inference time by **~3.6×** and peak VRAM by
> **~36%**.  This makes real-time or resource-constrained deployment viable
> without any accuracy trade-off.

**Secondary contribution: depth sharpening via adaptive resolution**  
Although resolution does not affect pose accuracy, higher-resolution passes
may produce sharper depth maps. The adaptive strategy (global 224 px + targeted
518 px patches on uncertain regions) is evaluated here as a *depth-quality*
improvement, not a pose-accuracy improvement.

| Method | Expected ATE | Expected time | Use case |
|--------|-------------|--------------|----------|
| 224 px baseline | same as 518 px | fastest | **recommended default** |
| 518 px baseline | same as 224 px | 3.6× slower | only if raw depth sharpness needed |
| Adaptive (224+patches) | same as 224 px | 1.5–2× 224 | sharper depth in uncertain regions |
| Progressive (224→336→518) | same as 224 px | ~3× 224 | depth pyramid, diminishing returns |

## 0 — Setup

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def git(*args):
    subprocess.check_call(["git"] + list(args))

pip("--upgrade", "numpy")
pip("plotly", "pandas", "pillow", "scipy", "matplotlib")

if not os.path.isdir("vggt"):
    git("clone", "--depth", "1", "https://github.com/facebookresearch/vggt.git")
    pip("-e", "vggt")

if not os.path.isdir("vggt-eval"):
    git("clone", "--depth", "1", "https://github.com/insha-py/vggt-eval.git")
else:
    git("-C", "vggt-eval", "pull", "--ff-only")

sys.path.insert(0, "vggt-eval")
sys.path.insert(0, "vggt")

print("Packages installed. Restarting kernel ...")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Restart the kernel manually, then run from the Imports cell.")

In [ ]:
import os, sys, time
for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.tum_vi                   import TUMVIDataset
from src.pipeline                 import VGGTPipeline, load_images_from_dir, run_vggt_inference
from src.improvements.adaptive    import AdaptiveResolutionVGGT
from src.improvements.progressive import ProgressiveRefinement
from src.metrics                  import (
    compute_ate, compute_rpe, compute_rotation_errors,
    MemoryProfiler, Timer,
)
from src.resolution_sweep         import ResolutionSweeper, DEFAULT_RESOLUTIONS

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1 — Download TUM-VI `room1`

In [ ]:
SEQUENCE     = "room1"
N_FRAMES     = 8
DOWNLOAD_DIR = "/tmp/tumvi"

ds   = TUMVIDataset(sequence=SEQUENCE, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
ds.download()
data = ds.load()

image_dir     = data["image_dir"]
gt_extrinsics = data["gt_extrinsics"]   # (N, 3, 4) — now in camera frame after R_CAM_IMU fix
N = len(gt_extrinsics)
print(f"Loaded {N} frames")

## 2 — Load Model and Run All Methods

In [ ]:
pipe = VGGTPipeline()
pipe.load_model()

def run_at_res(res):
    imgs, _, _ = load_images_from_dir(image_dir, target_size=res, max_frames=N_FRAMES)
    with MemoryProfiler() as mem, Timer() as tmr:
        out = run_vggt_inference(pipe.model, imgs, pipe.device, pipe.dtype, resolution=res)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out, tmr.elapsed, mem.peak_mb

# --- 224 px (recommended low-cost operating point) ---
print("[1/4] 224 px baseline ...")
out_224, t_224, mb_224 = run_at_res(224)

# --- 518 px (native training resolution, high cost) ---
print("[2/4] 518 px baseline ...")
out_518, t_518, mb_518 = run_at_res(518)

# --- Adaptive: global 224 px + 518 px patch refinement for uncertain depth ---
print("[3/4] Adaptive (224 global + 518 px patches) ...")
adaptive = AdaptiveResolutionVGGT(
    global_res=224, patch_res=518, conf_threshold=0.3,
    patch_size_px=64, max_patches=4,
)
adaptive._model  = pipe.model
adaptive._device = pipe.device
adaptive._dtype  = pipe.dtype

t0 = time.perf_counter()
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
out_adapt = adaptive.run(image_dir=image_dir, max_frames=N_FRAMES)
t_adapt   = time.perf_counter() - t0
mb_adapt  = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"  patches_used={out_adapt['patches_used']}")

# --- Progressive: coarse-to-fine pyramid ---
print("[4/4] Progressive (224 → 336 → 518) ...")
progressive = ProgressiveRefinement(pyramid=[224, 336, 518], blend_mode="conf_weighted")
progressive._model  = pipe.model
progressive._device = pipe.device
progressive._dtype  = pipe.dtype

t0 = time.perf_counter()
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
out_prog = progressive.run(image_dir=image_dir, max_frames=N_FRAMES)
t_prog   = time.perf_counter() - t0
mb_prog  = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\n224 px: {t_224:.1f}s  {mb_224:.0f} MB")
print(f"518 px: {t_518:.1f}s  {mb_518:.0f} MB  ({t_518/t_224:.1f}x slower, {100*(mb_518-mb_224)/mb_518:.0f}% more VRAM)")
print(f"Adaptive: {t_adapt:.1f}s  {mb_adapt:.0f} MB")
print(f"Progressive: {t_prog:.1f}s  {mb_prog:.0f} MB")

## 3 — Pose Accuracy: All Methods vs GT

In [ ]:
def metrics_row(ext, label, time_s, peak_mb):
    gt = gt_extrinsics[:len(ext)]
    ate = compute_ate(ext, gt, align=True, with_scale=True)
    rpe = compute_rpe(ext, gt, step=1)
    rot = compute_rotation_errors(ext, gt)
    return dict(
        Method       = label,
        ATE_mean     = round(ate["mean"],       4),
        ATE_rmse     = round(ate["rmse"],       4),
        RPE_t_mean   = round(rpe["trans_mean"], 4),
        RPE_r_mean   = round(rpe["rot_mean"],   2),
        Rot_mean_deg = round(rot["mean"],       2),
        time_s       = round(time_s,  2),
        peak_mb      = round(peak_mb, 0),
    )

rows = [
    metrics_row(out_224["extrinsic"],   "224 px  ← recommended", t_224,   mb_224),
    metrics_row(out_518["extrinsic"],   "518 px  (native)",       t_518,   mb_518),
    metrics_row(out_adapt["extrinsic"], "Adaptive 224+518",        t_adapt, mb_adapt),
    metrics_row(out_prog["extrinsic"],  "Progressive 224→336→518", t_prog,  mb_prog),
]

df = pd.DataFrame(rows)
print(f"\n=== Pose Accuracy: TUM-VI {SEQUENCE} ({N} frames) ===")
print(df.to_string(index=False))

os.makedirs("results", exist_ok=True)
df.to_csv(f"results/phase5_efficiency_{SEQUENCE}.csv", index=False)
print(f"\nSaved to results/phase5_efficiency_{SEQUENCE}.csv")

## 4 — Efficiency Contribution: The Core Thesis Plot

In [ ]:
# Load the full resolution sweep from 04b to get all 6 resolution points
sweep_csv = f"results/phase5b_resolution_sweep_singlepass_{SEQUENCE}.csv"
if os.path.isfile(sweep_csv):
    df_sweep = pd.read_csv(sweep_csv)
else:
    # Fall back to the two baselines we just ran
    df_sweep = pd.DataFrame([
        {"resolution": 224, "ate_mean": rows[0]["ATE_mean"],
         "time_s": t_224, "peak_mb": mb_224},
        {"resolution": 518, "ate_mean": rows[1]["ATE_mean"],
         "time_s": t_518, "peak_mb": mb_518},
    ])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("ATE vs Resolution", "Inference Time vs Resolution",
                    "Peak VRAM vs Resolution"),
)

kw = dict(mode="lines+markers", marker=dict(size=9))

fig.add_trace(go.Scatter(x=df_sweep["resolution"], y=df_sweep["ate_mean"],
                         line=dict(color="royalblue", width=2), name="ATE", **kw),
              row=1, col=1)
fig.add_trace(go.Scatter(x=df_sweep["resolution"], y=df_sweep["time_s"],
                         line=dict(color="steelblue", width=2), name="Time", **kw),
              row=1, col=2)
fig.add_trace(go.Scatter(x=df_sweep["resolution"], y=df_sweep["peak_mb"],
                         line=dict(color="darkorange", width=2), name="VRAM", **kw),
              row=1, col=3)

for col in (1, 2, 3):
    fig.add_vline(x=224, line_dash="dot",  line_color="green",
                  annotation_text="224 px" if col == 1 else "",
                  annotation_position="top right", row=1, col=col)
    fig.add_vline(x=518, line_dash="dash", line_color="grey",
                  annotation_text="518 px (native)" if col == 1 else "",
                  annotation_position="top left", row=1, col=col)

fig.update_xaxes(title_text="Resolution (px)")
fig.update_yaxes(title_text="ATE (m)",  row=1, col=1)
fig.update_yaxes(title_text="Time (s)", row=1, col=2)
fig.update_yaxes(title_text="VRAM (MB)",row=1, col=3)

fig.update_layout(
    height=430,
    title_text="The Efficiency Contribution: flat ATE, rising cost — 224 px is optimal",
    showlegend=False,
)
fig.show()

## 5 — Pose Accuracy Bar Chart

In [ ]:
colors = ["#2ecc71", "#e74c3c", "#f39c12", "#3498db"]  # green=recommended, red=expensive

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("ATE mean (m) — lower is better",
                                    "Inference time (s) — lower is better"))
for i, row in df.iterrows():
    fig.add_trace(go.Bar(x=[row["Method"]], y=[row["ATE_mean"]],
                         marker_color=colors[i], showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=[row["Method"]], y=[row["time_s"]],
                         marker_color=colors[i], showlegend=False), row=1, col=2)

fig.update_layout(height=420, title_text=f"All Methods — TUM-VI {SEQUENCE}", bargap=0.3)
fig.show()

## 6 — Depth Map Quality: 224 px vs 518 px
Resolution does not affect pose accuracy, but may affect depth map sharpness.
This is the only remaining justification for higher-resolution passes.

In [ ]:
import torch.nn.functional as F
import torch as th

FRAME_IDX = 0
DISPLAY_SIZE = 256

def to_display(arr, size=DISPLAY_SIZE):
    t = th.from_numpy(arr.astype("float32")).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
    return t.squeeze().numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pairs = [
    (out_224["depth_map"][FRAME_IDX, :, :, 0],  out_224["depth_conf"][FRAME_IDX],
     "Depth 224 px",           "Conf 224 px"),
    (out_518["depth_map"][FRAME_IDX, :, :, 0],  out_518["depth_conf"][FRAME_IDX],
     "Depth 518 px (native)",   "Conf 518 px"),
    (out_adapt["depth_map"][FRAME_IDX, :, :, 0], out_adapt["depth_conf"][FRAME_IDX],
     "Depth Adaptive",         "Conf Adaptive"),
    (out_prog["depth_map"][FRAME_IDX, :, :, 0],  out_prog["depth_conf"][FRAME_IDX],
     "Depth Progressive",      "Conf Progressive"),
]
for col, (d, c, dt, ct) in enumerate(pairs):
    axes[0, col].imshow(to_display(d), cmap="plasma")
    axes[0, col].set_title(dt, fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(to_display(c), cmap="viridis", vmin=0)
    axes[1, col].set_title(ct, fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(f"Depth & Confidence — Frame {FRAME_IDX}  "
             "(visual quality comparison — pose accuracy is equal)", y=1.02)
plt.tight_layout()
plt.show()

## 7 — Depth Confidence Statistics

In [ ]:
methods_conf = [
    ("224 px  ← recommended", out_224["depth_conf"]),
    ("518 px  (native)",       out_518["depth_conf"]),
    ("Adaptive 224+518",        out_adapt["depth_conf"]),
    ("Progressive 224→336→518", out_prog["depth_conf"]),
]

print(f"{'Method':<28}  {'Mean conf':>10}  {'Median conf':>12}  {'<0.3 frac':>10}")
print("-" * 68)
for name, conf in methods_conf:
    print(f"{name:<28}  {conf.mean():>10.4f}  "
          f"{float(np.median(conf)):>12.4f}  "
          f"{float((conf < 0.3).mean()):>10.4f}")

## 8 — Accuracy-Efficiency Scatter

In [ ]:
fig = px.scatter(
    df, x="time_s", y="ATE_mean",
    text="Method", color="Method",
    color_discrete_sequence=colors,
    title="Accuracy-Efficiency Trade-off (lower-left = better)\n"
          "Key finding: all methods land at the same ATE — choose the fastest",
    labels={"time_s": "Inference time (s)", "ATE_mean": "ATE mean (m)"},
    size=[12] * len(df),
)
fig.update_traces(textposition="top center")
fig.update_layout(showlegend=False)
fig.show()

## 9 — Thesis Contribution Summary

### Primary finding — Resolution invariance enables free efficiency gains

VGGT's pose accuracy is **invariant to input resolution** across 224–518 px,
confirmed on TUM-VI room1 in both chunked (40 frames) and single-pass (8 frames)
modes.  This is a non-trivial empirical result: VGGT's ViT tokeniser produces
equally good pose estimates from 7× fewer input pixels.

**Quantified savings (224 px vs 518 px on Tesla T4):**

| Metric | 224 px | 518 px | Saving |
|--------|--------|--------|-------|
| ATE mean | same | same | 0% (no trade-off) |
| Inference time | ~1.2 s | ~3.8 s | **3.2× faster** |
| Peak VRAM | 6 996 MB | 9 529 MB | **36% less** |

### Secondary finding — Depth sharpening via adaptive resolution

Since pose accuracy cannot be improved (it is already optimal at 224 px),
the adaptive strategy's value lies in depth map quality for applications
that require dense reconstruction (mapping, object modelling).  The trade-off:
~1.5–2× the 224 px inference time for targeted depth refinement.

### Recommendation

| Use case | Recommended setting |
|----------|--------------------|
| Pose estimation only | **224 px** |
| Pose + coarse depth | **224 px** |
| Pose + high-quality depth | **Adaptive (224+patches)** |
| Pose + depth pyramid | Progressive (3× cost, marginal gain) |
| Hard VRAM constraint (<7 GB) | **224 px** (only option below 7 GB) |